# P-Task5: Order Discount Pipeline

**Problem:** online store ke paas customer orders ki raw list hai. Category ke hisaab se discount lagana hai,
jin orders ki final amount minimum threshold se kam ho unhein filter karna hai aur end par total revenue
ke saath processed summary deni hai. **Poore code mein koi `for` loop nahi.**

**Concepts used:** `lambda`, `map()`, `filter()`, `functools.reduce()`, `sorted()` with lambda key.

**Pipeline:**
```
raw orders -> discount lagao (map) -> minimum se kam hatao (filter) -> total (reduce) -> summary
```

## 1. Data aur rules

In [1]:
from functools import reduce

orders = [
    {"order_id": 1, "customer": "Ayesha", "category": "electronics", "amount": 12000},
    {"order_id": 2, "customer": "Bilal",  "category": "clothing",    "amount": 2500},
    {"order_id": 3, "customer": "Fatima", "category": "books",       "amount": 450},
    {"order_id": 4, "customer": "Usman",  "category": "grocery",     "amount": 300},
    {"order_id": 5, "customer": "Hina",   "category": "clothing",    "amount": 4800},
    {"order_id": 6, "customer": "Zain",   "category": "electronics", "amount": 800},
    {"order_id": 7, "customer": "Sara",   "category": "books",       "amount": 1500},
    {"order_id": 8, "customer": "Omar",   "category": "toys",        "amount": 1200},
    {"order_id": 9, "customer": "Kiran",  "category": "grocery",     "amount": 2100},
]

# category : discount percent
discount_rules = {"electronics": 10, "clothing": 20, "books": 15, "grocery": 5}
MIN_AMOUNT = 500            # discount ke baad is se kam wale orders reject

len(orders)

9

## 2. Step 1: discount lagana (`map`)
`toys` ka rule dictionary mein nahi hai, isliye `.get(category, 0)` se uska discount 0 liya.
Yahan named function use kiya kyun ke logic thora lamba hai. Chhote kaam ke liye lambda theek hota hai.

In [2]:
def apply_discount(order):
    percent = discount_rules.get(order["category"], 0)
    final = order["amount"] * (100 - percent) / 100
    return {**order, "discount_percent": percent, "final_amount": final}


discounted = list(map(apply_discount, orders))
discounted[0]

{'order_id': 1, 'customer': 'Ayesha', 'category': 'electronics', 'amount': 12000, 'discount_percent': 10, 'final_amount': 10800.0}

## 3. Step 2: minimum se kam orders hatana (`filter`)

In [3]:
valid_orders = list(filter(lambda o: o["final_amount"] >= MIN_AMOUNT, discounted))
rejected = list(filter(lambda o: o["final_amount"] < MIN_AMOUNT, discounted))

print("Valid   :", len(valid_orders))
print("Rejected:", list(map(lambda o: (o["order_id"], o["final_amount"]), rejected)))

Valid   : 7
Rejected: [(3, 382.5), (4, 285.0)]


## 4. Step 3: total revenue (`reduce`)
`reduce` list ko ek number mein samet deta hai: `acc` ab tak ka total, `o` agla order.

In [4]:
total_revenue = reduce(lambda acc, o: acc + o["final_amount"], valid_orders, 0)
total_discount = reduce(lambda acc, o: acc + (o["amount"] - o["final_amount"]), valid_orders, 0)

print(f"Total revenue  : Rs {total_revenue:,.2f}")
print(f"Total discount : Rs {total_discount:,.2f}")

Total revenue  : Rs 21,830.00
Total discount : Rs 3,070.00


## 5. Category wise revenue (bina loop ke)
Pehle unique categories nikaali (set), phir har category ke liye filter + reduce.

In [5]:
categories = sorted(set(map(lambda o: o["category"], valid_orders)))

revenue_by_category = dict(map(
    lambda cat: (cat, reduce(lambda acc, o: acc + o["final_amount"],
                             filter(lambda o: o["category"] == cat, valid_orders), 0)),
    categories,
))
revenue_by_category

{'books': 1275.0, 'clothing': 5840.0, 'electronics': 11520.0, 'grocery': 1995.0, 'toys': 1200.0}

## 6. Final summary

In [6]:
best_first = sorted(valid_orders, key=lambda o: o["final_amount"], reverse=True)

order_lines = map(
    lambda o: f"  #{o['order_id']:<3}{o['customer']:<8}{o['category']:<13}"
              f"Rs {o['amount']:>7,} -{o['discount_percent']:>2}%  = Rs {o['final_amount']:>9,.2f}",
    best_first,
)
category_lines = map(lambda kv: f"  {kv[0]:<13}Rs {kv[1]:>10,.2f}", revenue_by_category.items())

summary = {
    "orders_received": len(orders),
    "orders_processed": len(valid_orders),
    "orders_rejected": len(rejected),
    "total_revenue": round(total_revenue, 2),
    "biggest_order": best_first[0]["order_id"],
}

print("ORDER DISCOUNT SUMMARY")
print("=" * 62)
print("\n".join(order_lines))
print("-" * 62)
print("Category wise revenue:")
print("\n".join(category_lines))
print("-" * 62)
print(summary)

ORDER DISCOUNT SUMMARY
  #1  Ayesha  electronics  Rs  12,000 -10%  = Rs 10,800.00
  #5  Hina    clothing     Rs   4,800 -20%  = Rs  3,840.00
  #2  Bilal   clothing     Rs   2,500 -20%  = Rs  2,000.00
  #9  Kiran   grocery      Rs   2,100 - 5%  = Rs  1,995.00
  #7  Sara    books        Rs   1,500 -15%  = Rs  1,275.00
  #8  Omar    toys         Rs   1,200 - 0%  = Rs  1,200.00
  #6  Zain    electronics  Rs     800 -10%  = Rs    720.00
--------------------------------------------------------------
Category wise revenue:
  books        Rs   1,275.00
  clothing     Rs   5,840.00
  electronics  Rs  11,520.00
  grocery      Rs   1,995.00
  toys         Rs   1,200.00
--------------------------------------------------------------
{'orders_received': 9, 'orders_processed': 7, 'orders_rejected': 2, 'total_revenue': 21830.0, 'biggest_order': 1}


## Lambda vs named function
| | lambda | named function (`def`) |
|---|---|---|
| Kab | chhota, ek line ka kaam (key, filter condition) | lamba ya baar baar use hone wala logic |
| Limit | sirf ek expression, statements nahi | koi limit nahi, docstring bhi de sakte hain |

`apply_discount` mein `def` isliye use kiya kyun ke wo ek expression se bada tha.

## Conclusion
9 orders mein se 7 process hue aur 2 minimum threshold se kam hone ki wajah se reject ho gaye. Poora pipeline
`map`, `filter` aur `reduce` se chala, kisi jagah `for` loop use nahi hua.